# Student Performance Prediction using AI/ML

## 1. Introduction

This project predicts whether a student will **Pass** or **Fail** using a real-world dataset of **40,000 student records**.

### Problem Statement
Educators need automated systems to identify at-risk students early. This project uses machine learning to analyze academic and behavioral factors to predict student outcomes.

### Objectives
- Load and preprocess a real Excel dataset with 40,000 records
- Perform Exploratory Data Analysis (EDA)
- Train Logistic Regression and Random Forest models
- Compare model performance using accuracy, precision, recall, and confusion matrix
- Build a custom prediction function

### Methodology
1. Data Collection → Preprocessing → EDA → Model Training → Evaluation → Prediction

### Dataset Source
File: `student_performance_prediction dataset 2.xlsx` — 40,000 student records, 7 columns.

---

### Advantages
- Uses real-world data (40,000 records) for more realistic modeling
- Identifies the most important academic factors affecting pass/fail outcomes
- Provides a reusable prediction function for new student data

### Future Scope
- Collect more behaviorally diverse data to improve model accuracy
- Integrate deep learning models (Neural Networks)
- Deploy as a web API for school management systems

## 2. Dataset
Load the real Excel dataset and inspect its structure.

In [ ]:
# Import required libraries
import pandas as pd                  # Data manipulation
import numpy as np                   # Numerical operations
import matplotlib.pyplot as plt      # Plotting
import seaborn as sns                # Advanced visualizations
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

In [ ]:
# Load the real Excel dataset
df = pd.read_excel('student_performance_prediction dataset 2.xlsx')

print(f'Dataset Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 3. Preprocessing
Handle missing values, encode categorical variables, and prepare the data for modeling.

In [ ]:
# Check for missing values in each column
print('Null values per column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

In [ ]:
# Handle missing values: fill numeric columns with their mean
df['Study Hours per Week'].fillna(df['Study Hours per Week'].mean(), inplace=True)
df['Attendance Rate'].fillna(df['Attendance Rate'].mean(), inplace=True)
df['Previous Grades'].fillna(df['Previous Grades'].mean(), inplace=True)

# Encode binary categorical: Yes=1, No=0
le = LabelEncoder()
df['Participation in Extracurricular Activities'] = le.fit_transform(df['Participation in Extracurricular Activities'])

# Encode target column: 'Passed' -> Yes=1, No=0
df['Passed'] = df['Passed'].map({'Yes': 1, 'No': 0})

# Ordinal encode Parent Education Level: High School < Associate < Bachelor < Master < PhD
edu_order = {'High School': 1, 'Associate': 2, 'Bachelor': 3, 'Master': 4, 'PhD': 5}
df['Parent Education Level'] = df['Parent Education Level'].map(edu_order)

# Drop Student ID (not a feature)
df.drop(columns=['Student ID'], inplace=True)

print('Encoding complete. Pass/Fail distribution:')
print(df['Passed'].value_counts())
df.head()

## 4. EDA (Exploratory Data Analysis)
Visualize distributions, correlations, and feature relationships.

In [ ]:
# Dataset Info
print('Dataset Information:')
df.info()
print('\nDescriptive Statistics:')
df.describe()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(9, 6))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Attendance Rate vs Passed
plt.figure(figsize=(8, 5))
sns.boxplot(x='Passed', y='Attendance Rate', data=df, palette='Set2')
plt.title('Attendance Rate vs Performance (Pass/Fail)')
plt.xticks([0, 1], ['Fail', 'Pass'])
plt.show()

In [ ]:
# Study Hours per Week vs Passed
plt.figure(figsize=(8, 5))
sns.violinplot(x='Passed', y='Study Hours per Week', data=df, palette='pastel')
plt.title('Study Hours per Week vs Performance')
plt.xticks([0, 1], ['Fail', 'Pass'])
plt.show()

In [ ]:
# Previous Grades vs Passed
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Previous Grades', y='Passed', data=df.sample(500), hue='Passed', palette='deep', alpha=0.6)
plt.title('Previous Grades vs Performance')
plt.yticks([0, 1], ['Fail', 'Pass'])
plt.show()

In [ ]:
# Pass/Fail count distribution
plt.figure(figsize=(5, 4))
df['Passed'].value_counts().plot(kind='bar', color=['#ef4444', '#22c55e'], edgecolor='white')
plt.xticks([0, 1], ['Fail', 'Pass'], rotation=0)
plt.title('Pass vs Fail Distribution')
plt.ylabel('Count')
plt.show()

## 5. Model Training
Split data and train both models.

In [ ]:
# Define features (X) and target (y)
X = df.drop('Passed', axis=1)
y = df['Passed']

# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training samples: {len(X_train)}')
print(f'Testing samples:  {len(X_test)}')

In [ ]:
# Train Logistic Regression
lr_model = LogisticRegression(max_iter=500, random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
print('Logistic Regression training complete!')

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
print('Random Forest training complete!')

## 6. Model Evaluation
Compare both models using all evaluation metrics.

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    """Print evaluation metrics and plot confusion matrix."""
    print(f'=== {model_name} ===')
    print(f'Accuracy:  {accuracy_score(y_true, y_pred):.4f}')
    print(f'Precision: {precision_score(y_true, y_pred):.4f}')
    print(f'Recall:    {recall_score(y_true, y_pred):.4f}')
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['Fail', 'Pass']))
    # Confusion Matrix plot
    plt.figure(figsize=(5, 4))
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Fail', 'Pass'], yticklabels=['Fail', 'Pass'])
    plt.title(f'Confusion Matrix — {model_name}')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.tight_layout(); plt.show()
    print()

evaluate_model(y_test, lr_pred, 'Logistic Regression')
evaluate_model(y_test, rf_pred, 'Random Forest')

In [ ]:
# Compare Model Accuracies visually
lr_acc = accuracy_score(y_test, lr_pred)
rf_acc = accuracy_score(y_test, rf_pred)

models = ['Logistic Regression', 'Random Forest']
accuracies = [lr_acc, rf_acc]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accuracies, color=['#6366f1', '#ec4899'], edgecolor='white', width=0.5)
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.04,
             f'{acc:.2%}', ha='center', color='white', fontsize=12, fontweight='bold')
plt.ylim(0, 1)
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()

winner = 'Logistic Regression' if lr_acc >= rf_acc else 'Random Forest'
print(f'\n✅ Better Model: {winner} ({max(lr_acc, rf_acc):.2%} accuracy)')

In [ ]:
# Random Forest Feature Importances
feature_names = X.columns.tolist()
importances = rf_model.feature_importances_

fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi_df = fi_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 4))
sns.barplot(x='Importance', y='Feature', data=fi_df, palette='viridis')
plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

print('\nFeature Importances:')
print(fi_df.to_string(index=False))

## 7. Prediction on New Student Data

> **Note on Accuracy:** This real dataset has near-identical average feature values for Pass and Fail students (avg study hours Pass: 9.92h, Fail: 10.02h), making it a challenging classification problem. Both models achieve ~50–53% accuracy, which is honest — the data does not have strong linear separability.

In [ ]:
def predict_student(study_hours_per_week, attendance_rate, previous_grades,
                    extracurricular, parent_edu_level):
    """
    Predict student Pass/Fail using the trained Logistic Regression model.
    
    Parameters:
        study_hours_per_week (float) : Weekly study hours (e.g. 12.5)
        attendance_rate (float)      : Attendance % (0-100)
        previous_grades (float)      : Previous exam marks (0-100)
        extracurricular (str)        : 'Yes' or 'No'
        parent_edu_level (str)       : 'High School', 'Associate', 'Bachelor', 'Master', 'PhD'
    """
    # Encode inputs (same as training)
    extra_enc = 1 if str(extracurricular).strip().lower() == 'yes' else 0
    edu_order = {'High School': 1, 'Associate': 2, 'Bachelor': 3, 'Master': 4, 'PhD': 5}
    parent_enc = edu_order.get(parent_edu_level, 3)
    
    # Build feature array
    features = np.array([[study_hours_per_week, attendance_rate, previous_grades,
                          extra_enc, parent_enc]])
    
    # Scale using the fitted scaler
    features_scaled = scaler.transform(features)
    
    # Predict
    prediction = lr_model.predict(features_scaled)[0]
    probability = lr_model.predict_proba(features_scaled)[0]
    
    result = 'PASS' if prediction == 1 else 'FAIL'
    confidence = probability[1] if prediction == 1 else probability[0]
    
    print(f'\nStudent Input:')
    print(f'  Study Hours/Week:    {study_hours_per_week}')
    print(f'  Attendance Rate:     {attendance_rate}%')
    print(f'  Previous Grades:     {previous_grades}')
    print(f'  Extracurricular:     {extracurricular}')
    print(f'  Parent Education:    {parent_edu_level}')
    print(f'\n🎯 Prediction: {result} (Confidence: {confidence:.1%})')
    return result

# ---- Test with sample data ----
predict_student(
    study_hours_per_week=15.0,
    attendance_rate=92.0,
    previous_grades=78.0,
    extracurricular='Yes',
    parent_edu_level='Bachelor'
)

## 8. Conclusion

We successfully built an AI/ML pipeline for student performance prediction using a **real-world dataset of 40,000 student records**.

### Key Findings
- **Previous Grades** (32.9%) and **Attendance Rate** (32.8%) are the most important factors — nearly equal contributors.
- **Study Hours per Week** (28.6%) is the third most important feature.
- The dataset's near-equal class distribution (~47.5% Pass, ~52.5% Fail) and similar feature means for both classes make this a genuinely hard classification problem.
- Both models achieved honest accuracy around **50–53%**, reflecting the true complexity of real-world student data.

### Models Trained
| Model | Accuracy |
|---|---|
| Logistic Regression | ~52.8% |
| Random Forest | ~50.3% |

### Real-World Insight
In practice, predicting student outcomes with high accuracy requires richer behavioral data — mental health scores, socioeconomic factors, teacher quality ratings, etc. This project demonstrates the full ML pipeline on authentic data.

---
**End of Project** | Dataset: 40,000 records | Models: Logistic Regression + Random Forest